# TruthLens Audio -- v9: close the voice-cloning-system generalization gap

**What v8 fixed:** false positives on genuine audio from unfamiliar recording conditions
(LibriSpeech: 0/10 -> 100%). That fix is intact and not touched here.

**What a systematic stress test just found:** scored v8 against one clip from each of the 59
distinct voice-conversion systems in ASVspoof2021-DF's VCC2020 track (a public research
challenge; none of these systems appear anywhere in v8's training data) -- v8 got **30/59
(50.8%), essentially chance**. v8 learned to recognize the specific voice-cloning artifact
signatures of the systems it was trained on (ASVspoof2019-LA's VC attacks, DEEP-VOICE,
In-The-Wild, the bank-fraud set), not general voice-cloning artifacts -- the same shortcut-learning
failure mode as before, just on the fake side instead of the genuine side this time.

**The fix:** add a new training source drawn from a wide spread of VCC2020 systems, split
**by system** (not by file) into ~40 systems used in training and ~19 systems **held out
entirely** -- never touched in train or val -- forming a new test set, T6. This mirrors exactly
the speaker-level splits already used for In-The-Wild/LibriSpeech, just on the axis that
actually matters here: which voice-conversion system generated the clip, not which speaker or
recording condition.

**What this will NOT do:** make the model generalize to every voice-cloning technique that
exists or ever will -- that is an open problem in the field, not something one more training
run solves. The realistic goal is: hold steady on T1-T5 (nothing regresses), and get T6
meaningfully above chance (the bar agreed before running this: matching v8 on T1-T5, T6 clearly
better than 50%) -- not 100%, and if it lands short of that, report it honestly rather than
running another loop immediately.

**Starting point:** fine-tunes from v8 (not from scratch). Resumable throughout, same
verified-Drive-save pattern proven across v4-v8.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

PROJECT_DIR = '/content/drive/MyDrive/TruthLens_Audio'
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

LABELS = ['genuine', 'deepfake']
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS)}

MANIFEST_DIR = f'{PROJECT_DIR}/manifests_v2'       # v2's manifest (ASVspoof + FakeAVCeleb ...)
MANIFEST_DIR_V4 = f'{PROJECT_DIR}/manifests_v4'    # has the DEEP-VOICE chunk rows
export_dir = f'{PROJECT_DIR}/export_v2'            # starting weights
CKPT_DIR_V5 = f'{PROJECT_DIR}/checkpoints_v5'      # only for the v5 reference column
CKPT_DIR_V7 = f'{PROJECT_DIR}/checkpoints_v7'      # v8's starting point + v7 reference column
MANIFEST_DIR_V7 = f'{PROJECT_DIR}/manifests_v7'   # where v7 saved the In-The-Wild manifest -- needed to restore it

RUN_NAME_V8 = 'v8'
CKPT_DIR_V8 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V8}'
LOG_DIR_V8 = f'{PROJECT_DIR}/logs_{RUN_NAME_V8}'
MANIFEST_DIR_V8 = f'{PROJECT_DIR}/manifests_{RUN_NAME_V8}'

RUN_NAME_V9 = 'v9'
CKPT_DIR_V9 = f'{PROJECT_DIR}/checkpoints_{RUN_NAME_V9}'
LOG_DIR_V9 = f'{PROJECT_DIR}/logs_{RUN_NAME_V9}'
MANIFEST_DIR_V9 = f'{PROJECT_DIR}/manifests_{RUN_NAME_V9}'
for d in [CKPT_DIR_V8, LOG_DIR_V8, MANIFEST_DIR_V8, MANIFEST_DIR_V7,
          CKPT_DIR_V9, LOG_DIR_V9, MANIFEST_DIR_V9]:
    os.makedirs(d, exist_ok=True)

# Drive space check -- read-only, deletes nothing. A full Drive is what silently broke
# checkpoint saving before, so look at it up front.
free_gb = shutil.disk_usage(PROJECT_DIR).free / 1e9
print(f'Drive free space: {free_gb:.2f} GB')
big = []
for root, _, files in os.walk(PROJECT_DIR):
    for f in files:
        p = os.path.join(root, f)
        try:
            sz = os.path.getsize(p)
        except OSError:
            continue
        if sz > 300e6:
            big.append((sz, p))
for sz, p in sorted(big, reverse=True):
    print(f'  {sz/1e9:.2f} GB  {p}')
if free_gb < 2.5:
    print('\nWARNING: under 2.5 GB free. v7 saves a ~1.1 GB latest.pt each epoch plus best.pt '
          '(~0.4 GB), so saves may fail. The old checkpoints_v4/v5/v6 latest.pt files listed above '
          'only matter for resuming runs that are already finished -- freeing space is YOUR '
          'decision, nothing here deletes anything. The save check below will print '
          'drive_saved=False if a save fails.')

print('Setup complete.')


In [ ]:
!pip install -q transformers accelerate soundfile librosa scikit-learn kaggle pandas tqdm av


In [ ]:
import os, shutil
from pathlib import Path

# Safety net only -- restores below should already provide everything.
kaggle_cred_drive = Path(f'{PROJECT_DIR}/kaggle.json')

if not kaggle_cred_drive.exists():
    print('No cached Kaggle credential found in Drive -- please upload one now.')
    print('(Kaggle -> Settings -> API -> Create New Token)')
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, 'Expected a file to be uploaded'
    uploaded_name = next(iter(uploaded))
    shutil.copy(uploaded_name, kaggle_cred_drive)
    print('Cached to Drive for next time.')

raw = kaggle_cred_drive.read_text().strip()

if raw.startswith('KGAT_'):
    os.environ['KAGGLE_API_TOKEN'] = raw
    legacy_path = Path(os.path.expanduser('~/.kaggle/kaggle.json'))
    if legacy_path.exists():
        legacy_path.unlink()
    print('Detected new-style KGAT_ token -> set KAGGLE_API_TOKEN env var.')
else:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(kaggle_cred_drive, os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('Detected classic kaggle.json -> copied to ~/.kaggle/kaggle.json.')

print('Kaggle credentials ready.')


## Restore audio from the existing Drive backups (v2 data + DEEP-VOICE chunks)

In [ ]:
import subprocess

backup_path = f'{PROJECT_DIR}/data_backup.tar.gz'
assert os.path.exists(backup_path), f'No backup found at {backup_path} -- nothing to restore.'

subprocess.run(['cp', backup_path, '/content/data_backup.tar.gz'], check=True)
subprocess.run(['tar', '-xzf', '/content/data_backup.tar.gz', '-C', '/content'], check=True)
print('Restored v2 raw audio data (ASVspoof + FakeAVCeleb + WaveFake) from Drive backup.')


In [ ]:
import subprocess

V4_DATA_BACKUP = f'{PROJECT_DIR}/v4_data_backup.tar.gz'
assert os.path.exists(V4_DATA_BACKUP), f'{V4_DATA_BACKUP} not found'
subprocess.run(['cp', V4_DATA_BACKUP, '/content/v4_data_backup.tar.gz'], check=True)
try:
    # only the pre-chunked DEEP-VOICE clips are needed here (not the generic-TTS adarshsingh set)
    subprocess.run(['tar', '-xzf', '/content/v4_data_backup.tar.gz', '-C', '/content',
                    'deepvoice_chunks'], check=True)
except subprocess.CalledProcessError:
    subprocess.run(['tar', '-xzf', '/content/v4_data_backup.tar.gz', '-C', '/content'], check=True)

# The backup extracts one level above DATA_DIR; the manifest paths expect DATA_DIR/deepvoice_chunks.
link = f'{DATA_DIR}/deepvoice_chunks'
if os.path.exists('/content/deepvoice_chunks') and not os.path.exists(link):
    os.symlink('/content/deepvoice_chunks', link)
assert os.path.isdir(link) and os.listdir(link), 'deepvoice_chunks missing after restore'
print('DEEP-VOICE chunks ready:', len(os.listdir(link)), 'files')


## Helpers, model class, and the widened augmentation

Same idea as v7's phone-quality/noise augmentation, widened to cover more recording conditions,
and a synthetic reverb added. Applied with the SAME probabilities to every clip regardless of
label -- that symmetry is what stops "sounds distorted" from becoming a new shortcut for either
class.

In [ ]:
import random
import numpy as np
import pandas as pd
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
import soundfile as sf
import librosa
from pathlib import Path
from tqdm import tqdm

TARGET_SR = 16000
WINDOW_SAMPLES = TARGET_SR * 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU -- training will be far too slow. Runtime -> Change runtime type -> GPU.')

def load_mono_16k(path):
    # soundfile can't open some containers (notably .m4a) -- fall back to librosa (ffmpeg).
    try:
        audio, sr = sf.read(path, always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
        return audio.astype(np.float32)
    except Exception:
        audio, sr = librosa.load(str(path), sr=TARGET_SR, mono=True)
        return audio.astype(np.float32)

def rms_normalize(audio, target_dbfs=-20.0):
    rms = np.sqrt(np.mean(audio ** 2)) + 1e-9
    return audio * (10 ** (target_dbfs / 20) / rms)

def fixed_window(audio, length, train=False):
    if len(audio) >= length:
        start = np.random.randint(0, len(audio) - length + 1) if train else (len(audio) - length) // 2
        return audio[start:start + length]
    pad = length - len(audio)
    left = pad // 2
    return np.pad(audio, (left, pad - left), mode="reflect")

def _match_len(x, n):
    return x[:n] if len(x) >= n else np.pad(x, (0, n - len(x)))

def band_limit(audio):
    # v7 only ever used a fixed 8kHz round-trip. Randomizing the cutoff covers a wider range of
    # real-world bandwidth limits (different codecs/devices), not just one specific phone-quality
    # signature the model could learn to key on.
    target = random.choice([6000, 7000, 8000, 9000, 11025])
    low = librosa.resample(audio, orig_sr=TARGET_SR, target_sr=target)
    return _match_len(librosa.resample(low, orig_sr=target, target_sr=TARGET_SR), len(audio)).astype(np.float32)

def add_noise(audio):
    snr_db = random.uniform(12, 35)
    noise_power = np.mean(audio ** 2) / (10 ** (snr_db / 10))
    return (audio + np.random.randn(len(audio)) * np.sqrt(noise_power)).astype(np.float32)

def add_reverb(audio):
    # Cheap synthetic room reverb (exponentially-decaying noise impulse response) -- needs no
    # external IR files, just enough to teach the model that echo/room tone isn't a fake signal.
    decay_s = random.uniform(0.05, 0.25)
    n = int(decay_s * TARGET_SR)
    if n < 8:
        return audio
    ir = np.random.randn(n).astype(np.float32) * np.exp(-np.arange(n) / (n / 4))
    ir /= (np.abs(ir).sum() + 1e-9)
    ir[0] += 0.7  # keep the direct (dry) signal dominant so the voice stays intelligible
    wet = np.convolve(audio, ir, mode='full')[:len(audio)]
    return wet.astype(np.float32)

def augment_audio(audio):
    if random.random() < 0.30:
        audio = band_limit(audio)
    if random.random() < 0.30:
        audio = add_noise(audio)
    if random.random() < 0.30:
        audio = add_reverb(audio)
    return rms_normalize(audio)

class AudioDatasetV7(Dataset):
    def __init__(self, df, train, augment=False):
        self.rows = df.reset_index(drop=True)
        self.train = train
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        try:
            audio = load_mono_16k(row['path'])
            audio = rms_normalize(audio)
            audio = fixed_window(audio, WINDOW_SAMPLES, self.train)
            if self.augment:
                audio = augment_audio(audio)
        except Exception:
            audio = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        return torch.from_numpy(audio.astype(np.float32)), LABEL_TO_IDX[row['label']]

class AudioClassifier(nn.Module):
    def __init__(self, num_classes=2, freeze_feature_extractor=True):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        if freeze_feature_extractor:
            self.backbone.feature_extractor._freeze_parameters()
        hidden = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))

    def forward(self, waveform):
        out = self.backbone(waveform).last_hidden_state
        return self.head(out.mean(dim=1))

def collate(batch):
    return torch.stack([b[0] for b in batch]), torch.tensor([b[1] for b in batch])

LABELS = ['genuine', 'deepfake']
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS)}


## Upload the bank-fraud voice-changing set (used as one training source + one test set)

In [ ]:
from google.colab import files
import zipfile

print('Upload the zipped "audio" folder (voice_changing dataset).')
uploaded = files.upload()
zip_name = next(iter(uploaded))

eval_dir = '/content/real_eval_audio'
os.makedirs(eval_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(eval_dir)
print('Extracted to', eval_dir)


In [ ]:
import glob

eval_files = (
    glob.glob(f'{eval_dir}/**/*.mp3', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.m4a', recursive=True) +
    glob.glob(f'{eval_dir}/**/*.wav', recursive=True)
)
print(f'Found {len(eval_files)} real-world eval files.')
assert eval_files, 'No audio files found in the extracted zip -- check its folder structure.'


In [ ]:
from pathlib import Path

HELD_OUT_SPEAKER = ('USA', 'male')  # never used in training -- pure held-out eval target

realistic_finetune_rows = []
realistic_heldout_rows = []
for path in eval_files:
    parts = Path(path).parts
    country, gender = parts[-4], parts[-3]
    stem = Path(path).stem
    label = 'genuine' if stem == 'original' else 'deepfake'
    row = {'path': path, 'label': label, 'source': 'realistic_voice_changing',
           'speaker_id': f'{country}_{gender}', 'split_hint': None}
    if (country, gender) == HELD_OUT_SPEAKER:
        realistic_heldout_rows.append(row)
    else:
        realistic_finetune_rows.append(row)

print(f'Fine-tune rows (3 speakers): {len(realistic_finetune_rows)}')
print(f'Held-out eval rows (speaker {HELD_OUT_SPEAKER}): {len(realistic_heldout_rows)}')
assert realistic_finetune_rows and realistic_heldout_rows, 'Expected data on both sides of the speaker split.'


## In-The-Wild: restore the exact subset v7 already built and backed up (no re-download)

In [ ]:
import subprocess, zipfile

ITW_DIR = f'{DATA_DIR}/itw_subset'
ITW_BACKUP = f'{PROJECT_DIR}/itw_subset_backup.tar.gz'
ITW_MANIFEST = f'{MANIFEST_DIR_V7}/itw_manifest.csv'
ITW_MIRRORS = ['abdallamohamed312/in-the-wild-audio-deepfake',   # real/ + fake/ folders + meta.csv
               'abdallamohamed312/in-the-wild-dataset']          # backup copy (original release layout)
ITW_MIN_FILES = 30000     # the full dataset has 31,779 clips; fewer than this means a partial download
NESTED_MIN_BYTES = 1e9    # a zip bigger than this inside the download is treated as the real data archive

# caps per class per split (total ~8k clips)
ITW_CAP = {'train': 2500, 'val': 400, 'test': 800}
MAX_CLIP_SECONDS = 12

def kaggle_credentials_status():
    has_env = bool(os.environ.get('KAGGLE_API_TOKEN'))
    has_file = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
    print(f'Kaggle credentials in this session -> env token: {has_env}, kaggle.json: {has_file}')
    if not (has_env or has_file):
        print('  NO Kaggle credentials found. Run the "Kaggle credentials" cell near the top first.')

def list_dir(raw_dir):
    for name in sorted(os.listdir(raw_dir)):
        p = os.path.join(raw_dir, name)
        size = os.path.getsize(p) if os.path.isfile(p) else 0
        print(f'    {name}  {"(dir)" if os.path.isdir(p) else f"{size / 1e9:.2f} GB"}')

def extract_nested_archives(raw_dir):
    # This Kaggle mirror ships the data as a zip INSIDE the downloaded archive (a file literally named
    # 'download'), so --unzip leaves a big zip behind. Unpack it (no re-download needed).
    if not os.path.isdir(raw_dir):
        return
    for _ in range(3):
        if len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True)) >= ITW_MIN_FILES:
            return
        found = False
        for p in glob.glob(f'{raw_dir}/**/*', recursive=True):
            if os.path.isfile(p) and os.path.getsize(p) > NESTED_MIN_BYTES and zipfile.is_zipfile(p):
                print(f'  Unpacking nested archive {p} ({os.path.getsize(p) / 1e9:.2f} GB) ...')
                subprocess.run(['unzip', '-q', '-o', p, '-d', f'{raw_dir}/extracted'])
                os.remove(p)   # free the disk space; the extracted copy is what we use
                found = True
        if not found:
            return

def download_itw(raw_dir):
    kaggle_credentials_status()
    for ref in ITW_MIRRORS:
        shutil.rmtree(raw_dir, ignore_errors=True)   # never build on top of a partial download
        os.makedirs(raw_dir, exist_ok=True)
        print(f'Downloading {ref} (~9 GB, roughly 5-15 min, no progress bar) ...')
        r = subprocess.run(['kaggle', 'datasets', 'download', '-d', ref, '-p', raw_dir, '--unzip'],
                           capture_output=True, text=True)
        if r.returncode == 0:
            extract_nested_archives(raw_dir)
        n = len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True))
        if r.returncode == 0 and n >= ITW_MIN_FILES:
            print(f'  OK: {n} wav files')
            return ref
        print(f'  FAILED (exit code {r.returncode}, {n} wav files found). Kaggle said:')
        print('  ' + ((r.stderr or '') + (r.stdout or ''))[-800:].replace('\n', '\n  '))
        print('  Files left in the download folder:')
        list_dir(raw_dir)
    raise RuntimeError('Could not download In-The-Wild from any mirror -- read the Kaggle messages above.')

def meta_label_to_class(x):
    x = str(x).lower()
    if x in ('bona-fide', 'bonafide', 'bona_fide', 'real', 'genuine'):
        return 'genuine'
    if x in ('spoof', 'fake', 'deepfake'):
        return 'deepfake'
    return None

def restore_itw():
    subprocess.run(['cp', ITW_BACKUP, '/content/itw_subset_backup.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', '/content/itw_subset_backup.tar.gz', '-C', DATA_DIR], check=True)
    return pd.read_csv(ITW_MANIFEST)

def build_itw():
    raw_dir = f'{DATA_DIR}/itw_raw'
    extract_nested_archives(raw_dir)   # salvage a download already on disk instead of fetching 9 GB again
    if len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True)) < ITW_MIN_FILES:
        download_itw(raw_dir)

    all_wavs = glob.glob(f'{raw_dir}/**/*.wav', recursive=True)
    meta_path = glob.glob(f'{raw_dir}/**/meta.csv', recursive=True)
    assert meta_path, 'meta.csv (speaker info) not found'
    meta = pd.read_csv(meta_path[0])
    print('meta.csv columns:', meta.columns.tolist())
    print(meta.head(3).to_string())
    file_col = next(c for c in meta.columns if 'file' in c.lower())
    spk_col = next(c for c in meta.columns if 'speaker' in c.lower())
    label_col = next((c for c in meta.columns if 'label' in c.lower()), None)
    meta['_name'] = meta[file_col].astype(str).map(lambda x: Path(x).name if x.lower().endswith('.wav') else x + '.wav')
    speaker_of = dict(zip(meta['_name'], meta[spk_col].astype(str)))

    # label: from the real/ fake/ folders if present, otherwise from meta.csv (original release layout)
    folder_label = {}
    for p in all_wavs:
        parent = Path(p).parent.name.lower()
        if parent in ('real', 'fake'):
            folder_label[p] = 'genuine' if parent == 'real' else 'deepfake'
    if len(folder_label) < 0.9 * len(all_wavs):
        assert label_col is not None, 'No real/fake folders and no label column in meta.csv'
        name_to_label = dict(zip(meta['_name'], meta[label_col].map(meta_label_to_class)))
        folder_label = {p: name_to_label.get(Path(p).name) for p in all_wavs}
    rows = []
    for p, label in folder_label.items():
        spk = speaker_of.get(Path(p).name)
        if label is not None and spk is not None:
            rows.append({'path': p, 'label': label, 'speaker_id': spk})
    df = pd.DataFrame(rows)
    print(f'{len(df)} clips matched to a speaker and a label; classes:', df['label'].value_counts().to_dict())
    assert df['label'].nunique() == 2, 'Need both real and fake clips'
    if label_col is not None:  # sanity: our label vs meta's label agree
        name_to_meta = dict(zip(meta['_name'], meta[label_col].astype(str)))
        df['_meta_label'] = df['path'].map(lambda p: name_to_meta.get(Path(p).name))
        print(pd.crosstab(df['label'], df['_meta_label']))

    # split BY SPEAKER; both classes of a person stay in the same split
    speakers = sorted(df['speaker_id'].unique())
    rs = np.random.RandomState(42)
    rs.shuffle(speakers)
    n_test, n_val = max(1, round(0.2 * len(speakers))), max(1, round(0.1 * len(speakers)))
    spk_split = {}
    for i, s in enumerate(speakers):
        spk_split[s] = 'test' if i < n_test else ('val' if i < n_test + n_val else 'train')
    df['split'] = df['speaker_id'].map(spk_split)
    print(f'{len(speakers)} speakers -> train {sum(v == "train" for v in spk_split.values())}, '
          f'val {n_val}, test {n_test}')

    # subset: equal real/fake per split, capped
    parts = []
    for split, cap in ITW_CAP.items():
        d = df[df['split'] == split]
        n = min((d['label'] == 'genuine').sum(), (d['label'] == 'deepfake').sum(), cap)
        for label in LABELS:
            parts.append(d[d['label'] == label].sample(n, random_state=42))
    sub = pd.concat(parts).reset_index(drop=True)

    # convert to 16 kHz mono FLAC (compact + one uniform container for every clip)
    new_paths = []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc='converting to flac'):
        out_dir = f'{ITW_DIR}/{r["split"]}/{r["label"]}'
        os.makedirs(out_dir, exist_ok=True)
        out = f'{out_dir}/{i}.flac'
        if not os.path.exists(out):
            audio = load_mono_16k(r['path'])[:MAX_CLIP_SECONDS * TARGET_SR]
            sf.write(out, audio, TARGET_SR, format='FLAC')
        new_paths.append(out)
    sub['path'] = new_paths
    sub['source'] = 'inthewild'
    sub = sub[['path', 'label', 'source', 'speaker_id', 'split']]
    sub.to_csv(ITW_MANIFEST, index=False)

    subprocess.run(['tar', '-czf', '/content/itw_subset_backup.tar.gz', '-C', DATA_DIR, 'itw_subset'], check=True)
    size = os.path.getsize('/content/itw_subset_backup.tar.gz')
    free = shutil.disk_usage(PROJECT_DIR).free
    if size * 1.05 < free:
        shutil.copy2('/content/itw_subset_backup.tar.gz', ITW_BACKUP)
        ok = os.path.exists(ITW_BACKUP) and os.path.getsize(ITW_BACKUP) == size
        print(f'ITW backup to Drive: {size / 1e9:.2f} GB, saved={ok}')
    else:
        print(f'WARNING: Drive too full for the {size / 1e9:.2f} GB ITW backup ({free / 1e9:.2f} GB free) -- '
              'skipped. Next session will re-download In-The-Wild.')
    shutil.rmtree(raw_dir, ignore_errors=True)
    return sub

if os.path.exists(ITW_BACKUP) and os.path.exists(ITW_MANIFEST):
    print('Restoring In-The-Wild subset from Drive backup...')
    itw_df = restore_itw()
else:
    itw_df = build_itw()

print(itw_df.groupby(['split', 'label']).size().unstack())
print('speakers per split:', itw_df.groupby('split')['speaker_id'].nunique().to_dict())
train_spk = set(itw_df[itw_df.split == 'train'].speaker_id)
test_spk = set(itw_df[itw_df.split == 'test'].speaker_id)
assert not (train_spk & test_spk), 'speaker leak between train and test!'


## LibriSpeech: a new genuine-only source, chosen specifically because it's the
recording style v7 failed on (clean narration, no compression/phone artifacts). Split by
speaker, same principle as In-The-Wild: the test speakers are never trained on.

In [ ]:
import subprocess

LS_DIR = f'{DATA_DIR}/librispeech_subset'
LS_BACKUP = f'{PROJECT_DIR}/librispeech_subset_backup.tar.gz'
LS_MANIFEST = f'{MANIFEST_DIR_V8}/librispeech_manifest.csv'
LS_MIN_FILES = 25000   # the full dataset has 28,707 clips
LS_TRAIN_CAP = 2200    # capped -- this source only needs to teach "clean narration can be genuine",
LS_TEST_CAP = 60       # not become the dominant class; kept modest to bound training time.

def restore_librispeech():
    subprocess.run(['cp', LS_BACKUP, '/content/librispeech_subset_backup.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', '/content/librispeech_subset_backup.tar.gz', '-C', DATA_DIR], check=True)
    return pd.read_csv(LS_MANIFEST)

def build_librispeech():
    raw_dir = f'{DATA_DIR}/librispeech_raw'
    n_have = len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True))
    if n_have < LS_MIN_FILES:
        shutil.rmtree(raw_dir, ignore_errors=True)
        os.makedirs(raw_dir, exist_ok=True)
        print('Downloading LibriSpeech-Clean-EN (~6 GB, roughly 5-10 min)...')
        r = subprocess.run(['kaggle', 'datasets', 'download', '-d', 'junaedkhandakar/librispeech-clean-en',
                            '-p', raw_dir, '--unzip'], capture_output=True, text=True)
        n_have = len(glob.glob(f'{raw_dir}/**/*.wav', recursive=True))
        if r.returncode != 0 or n_have < LS_MIN_FILES:
            print('Kaggle said:')
            print(((r.stderr or '') + (r.stdout or ''))[-800:])
            raise RuntimeError(f'LibriSpeech download incomplete ({n_have} wav files found).')
    print(f'{n_have} wav files available.')

    clips_dir = next(Path(p) for p in glob.glob(f'{raw_dir}/**/clips', recursive=True))
    wavs = sorted(clips_dir.glob('*.wav'))
    # filename convention: <speaker>-<chapter>-<utterance>.wav
    df = pd.DataFrame({'path': [str(p) for p in wavs]})
    df['speaker_id'] = df['path'].map(lambda p: Path(p).stem.split('-')[0])
    df['label'] = 'genuine'
    print(f'{len(df)} clips, {df["speaker_id"].nunique()} speakers')

    speakers = sorted(df['speaker_id'].unique())
    rs = np.random.RandomState(7)   # different seed from ITW's -- independent split
    rs.shuffle(speakers)
    # 3-way speaker split, same principle as In-The-Wild's -- val MUST have its own speakers, not
    # just be absent, otherwise checkpoint selection (which reads val_df) is blind to whether the
    # LibriSpeech fix is actually working during training.
    n_test = max(10, round(0.05 * len(speakers)))
    n_val = max(5, round(0.05 * len(speakers)))
    test_speakers = set(speakers[:n_test])
    val_speakers = set(speakers[n_test:n_test + n_val])
    def _split_of(s):
        if s in test_speakers:
            return 'test'
        if s in val_speakers:
            return 'val'
        return 'train'
    df['split'] = df['speaker_id'].map(_split_of)

    LS_VAL_CAP = 150
    train_sub = df[df['split'] == 'train'].sample(min(LS_TRAIN_CAP, (df['split'] == 'train').sum()), random_state=7)
    val_sub = df[df['split'] == 'val'].sample(min(LS_VAL_CAP, (df['split'] == 'val').sum()), random_state=7)
    test_sub = df[df['split'] == 'test'].sample(min(LS_TEST_CAP, (df['split'] == 'test').sum()), random_state=7)
    sub = pd.concat([train_sub, val_sub, test_sub]).reset_index(drop=True)
    print(f'Subset: {len(train_sub)} train clips ({train_sub["speaker_id"].nunique()} speakers), '
          f'{len(val_sub)} val clips ({val_sub["speaker_id"].nunique()} speakers), '
          f'{len(test_sub)} test clips ({test_sub["speaker_id"].nunique()} speakers)')
    assert not (set(train_sub['speaker_id']) & set(val_sub['speaker_id']) & set(test_sub['speaker_id']))
    assert not (set(train_sub['speaker_id']) & set(test_sub['speaker_id'])), 'train/test speaker leak!'
    assert not (set(train_sub['speaker_id']) & set(val_sub['speaker_id'])), 'train/val speaker leak!'
    assert not (set(val_sub['speaker_id']) & set(test_sub['speaker_id'])), 'val/test speaker leak!'

    # copy the small subset out of the ~6GB raw download so it can be backed up cheaply
    os.makedirs(LS_DIR, exist_ok=True)
    new_paths = []
    for i, r in sub.iterrows():
        out = f'{LS_DIR}/{Path(r["path"]).name}'
        if not os.path.exists(out):
            shutil.copy2(r['path'], out)
        new_paths.append(out)
    sub['path'] = new_paths
    sub['source'] = 'librispeech'
    sub = sub[['path', 'label', 'source', 'speaker_id', 'split']]
    os.makedirs(MANIFEST_DIR_V8, exist_ok=True)
    sub.to_csv(LS_MANIFEST, index=False)

    subprocess.run(['tar', '-czf', '/content/librispeech_subset_backup.tar.gz', '-C', DATA_DIR,
                    'librispeech_subset'], check=True)
    size = os.path.getsize('/content/librispeech_subset_backup.tar.gz')
    free = shutil.disk_usage(PROJECT_DIR).free
    if size * 1.05 < free:
        shutil.copy2('/content/librispeech_subset_backup.tar.gz', LS_BACKUP)
        ok = os.path.exists(LS_BACKUP) and os.path.getsize(LS_BACKUP) == size
        print(f'LibriSpeech backup to Drive: {size / 1e9:.2f} GB, saved={ok}')
    else:
        print(f'WARNING: Drive too full for the {size / 1e9:.2f} GB backup ({free / 1e9:.2f} GB free) -- skipped.')
    shutil.rmtree(raw_dir, ignore_errors=True)
    return sub

if os.path.exists(LS_BACKUP) and os.path.exists(LS_MANIFEST):
    print('Restoring LibriSpeech subset from Drive backup...')
    librispeech_df = restore_librispeech()
else:
    librispeech_df = build_librispeech()

print(librispeech_df.groupby('split').size())
print('OK: genuine-only source, no fake counterpart needed (see assemble step for why).')


## VCC2020, system-diverse: the source that closes the T6 gap

One clip per VCC2020 voice-conversion system (59 total, from ASVspoof2021-DF's public research
eval set) was enough to reveal v8 only got 30/59 right -- chance level -- because it had never
seen most of these systems' artifact signatures. The fix isn't more clips from the *same* few
systems v8 already knows; it's exposure to more *distinct* systems.

Split **by system**, same principle as every other speaker-level split in this project: ~40
systems go into training (a handful of clips each), the other ~19 are **never touched** until
the T6 eval below. If val_df only saw training-system clips, checkpoint selection would be
blind to whether this is actually working -- so a held-back slice of *training-system* clips
(different files, same systems) goes into val, exactly like `ls_val` for LibriSpeech.


In [ ]:
import subprocess

VCC_DIR = f'{DATA_DIR}/vcc2020_subset'
VCC_BACKUP = f'{PROJECT_DIR}/vcc2020_subset_backup.tar.gz'
VCC_MANIFEST = f'{MANIFEST_DIR_V9}/vcc2020_manifest.csv'
VCC_META_LOCAL = f'{DATA_DIR}/vcc2020_trial_metadata.txt'
VCC_KAGGLE_REF = 'mohammedabdeldayem/avsspoof-2021'
VCC_META_KAGGLE_PATH = 'DF-keys-full/keys/DF/CM/trial_metadata.txt'
# File IDs are NOT guaranteed to live in part00 -- bonafide clips especially are scattered
# across the whole corpus (10 speakers x 225 clips, wide ID range), not clustered like the
# hand-picked sample of specific systems checked by hand earlier. Try each part in turn.
VCC_FLAC_PARTS = [
    'ASVspoof2021_DF_eval_part00/ASVspoof2021_DF_eval/flac',
    'ASVspoof2021_DF_eval_part01/ASVspoof2021_DF_eval/flac',
    'ASVspoof2021_DF_eval_part02/ASVspoof2021_DF_eval/flac',
]

VCC_MIN_SYSTEMS = 50         # sanity floor -- the real dataset has 59; far fewer means parsing broke
N_HELDOUT_SYSTEMS = 19       # never touched in train/val -- this is what T6 tests
VCC_TRAIN_PER_SYSTEM = 15    # per training system
VCC_VAL_PER_SYSTEM = 3       # per training system, disjoint files from the 15 above
VCC_HELDOUT_PER_SYSTEM = 10  # per held-out system -> T6

def download_vcc_file(kaggle_path, dest_path):
    # Idempotent: safe to re-run after a GPU-quota interruption, already-downloaded files are skipped.
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        return True
    dest_dir = os.path.dirname(dest_path)
    os.makedirs(dest_dir, exist_ok=True)
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d', VCC_KAGGLE_REF,
                        '-f', kaggle_path, '-p', dest_dir, '--force'],
                       capture_output=True, text=True)
    base = os.path.basename(kaggle_path)
    candidate = os.path.join(dest_dir, base)
    zip_candidate = candidate + '.zip'
    if os.path.exists(zip_candidate):
        subprocess.run(['unzip', '-q', '-o', zip_candidate, '-d', dest_dir])
        os.remove(zip_candidate)
    if os.path.exists(candidate) and os.path.getsize(candidate) > 0:
        if candidate != dest_path:
            os.replace(candidate, dest_path)
        return True
    return False

def download_vcc_flac(file_id, dest_path):
    # Try part00 first (fast path -- most spoof files land here), then fall back to
    # part01/part02 for files that don't. A silent, uncaught miss here is exactly what
    # produced empty val/heldout genuine slices last run -- never assume one part covers
    # everything.
    for part_dir in VCC_FLAC_PARTS:
        if download_vcc_file(f'{part_dir}/{file_id}.flac', dest_path):
            return True
    return False

def parse_vcc_metadata():
    # Manual line parsing, not pd.read_csv -- bonafide rows have fewer whitespace-separated
    # fields than spoof rows (no system-id/task/team columns), so a fixed column count breaks.
    rows = []
    with open(VCC_META_LOCAL) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 8:
                continue
            speaker, file_id, codec, source_dataset, system_id, label, trim, split_ = parts[:8]
            if source_dataset == 'vcc2020' and split_ == 'eval' and label in ('spoof', 'bonafide'):
                rows.append({'speaker': speaker, 'file_id': file_id, 'system_id': system_id, 'label': label})
    return pd.DataFrame(rows)

def build_vcc_diverse():
    if not os.path.exists(VCC_META_LOCAL):
        print('Downloading VCC2020 trial metadata (~62MB, one-time)...')
        ok = download_vcc_file(VCC_META_KAGGLE_PATH, VCC_META_LOCAL)
        assert ok, 'Failed to download trial_metadata.txt -- check Kaggle credentials/quota'

    meta = parse_vcc_metadata()
    spoof = meta[meta['label'] == 'spoof']
    # VCC2020's own bonafide clips are NOT used -- many sampled IDs are absent from all 3
    # download parts on this Kaggle mirror (verified by hand), unrelated to system diversity.
    # Genuine-side generalization is already covered by v8 (LibriSpeech/VoxCeleb); the assemble
    # step pairs these spoof clips with already-reliable In-The-Wild genuine clips instead.
    assert spoof['system_id'].nunique() >= VCC_MIN_SYSTEMS, 'Unexpectedly few VCC2020 systems found -- check parsing'

    systems = sorted(spoof['system_id'].unique())
    rs = np.random.RandomState(11)   # independent seed from ITW's (42) and LibriSpeech's (7)
    rs.shuffle(systems)
    heldout_systems = set(systems[:N_HELDOUT_SYSTEMS])
    train_systems = set(systems[N_HELDOUT_SYSTEMS:])
    print(f'{len(systems)} VCC2020 voice-conversion systems -> {len(train_systems)} in training, '
          f'{len(heldout_systems)} held out entirely for T6.')

    def sample_rows(pool, n, seed):
        n = min(n, len(pool))
        return pool.sample(n, random_state=seed)

    train_parts, val_parts, heldout_parts = [], [], []
    for sys_id in sorted(train_systems):
        pool = spoof[spoof['system_id'] == sys_id]
        tr = sample_rows(pool, VCC_TRAIN_PER_SYSTEM, seed=11)
        remaining = pool[~pool['file_id'].isin(tr['file_id'])]
        va = sample_rows(remaining, VCC_VAL_PER_SYSTEM, seed=12)
        train_parts.append(tr)
        val_parts.append(va)
    for sys_id in sorted(heldout_systems):
        pool = spoof[spoof['system_id'] == sys_id]
        heldout_parts.append(sample_rows(pool, VCC_HELDOUT_PER_SYSTEM, seed=13))

    spoof_train = pd.concat(train_parts, ignore_index=True)
    spoof_val = pd.concat(val_parts, ignore_index=True)
    spoof_heldout = pd.concat(heldout_parts, ignore_index=True)
    assert not (set(spoof_train['system_id']) & set(spoof_heldout['system_id'])), 'system leak: train/heldout!'
    assert not (set(spoof_val['system_id']) & set(spoof_heldout['system_id'])), 'system leak: val/heldout!'
    assert not (set(spoof_train['file_id']) & set(spoof_val['file_id'])), 'file leak: train/val!'

    def tag(df, label, split_name):
        d = df.copy()
        d['label'] = label
        d['split'] = split_name
        return d

    all_needed = pd.concat([
        tag(spoof_train, 'deepfake', 'train'),
        tag(spoof_val, 'deepfake', 'val'),
        tag(spoof_heldout, 'deepfake', 'heldout'),
    ], ignore_index=True)

    os.makedirs(VCC_DIR, exist_ok=True)
    paths, failed = [], 0
    for _, r in tqdm(all_needed.iterrows(), total=len(all_needed), desc='downloading VCC2020 clips'):
        local = f'{VCC_DIR}/{r["file_id"]}.flac'
        ok = download_vcc_flac(r["file_id"], local)
        paths.append(local if ok else None)
        if not ok:
            failed += 1
    all_needed['path'] = paths
    n_before = len(all_needed)
    all_needed = all_needed.dropna(subset=['path']).copy()
    if failed:
        print(f'WARNING: {failed}/{n_before} spoof clips failed to download (network/quota) -- '
              f'proceeding with the rest. (No bonafide download needed -- paired with ITW genuine in assemble.)')

    all_needed['source'] = 'vcc2020div'
    sub = all_needed[['path', 'label', 'source', 'system_id', 'split']]
    os.makedirs(MANIFEST_DIR_V9, exist_ok=True)
    sub.to_csv(VCC_MANIFEST, index=False)

    subprocess.run(['tar', '-czf', '/content/vcc2020_subset_backup.tar.gz', '-C', DATA_DIR, 'vcc2020_subset'], check=True)
    size = os.path.getsize('/content/vcc2020_subset_backup.tar.gz')
    free = shutil.disk_usage(PROJECT_DIR).free
    if size * 1.05 < free:
        shutil.copy2('/content/vcc2020_subset_backup.tar.gz', VCC_BACKUP)
        ok = os.path.exists(VCC_BACKUP) and os.path.getsize(VCC_BACKUP) == size
        print(f'VCC2020-diverse backup to Drive: {size/1e9:.2f} GB, saved={ok}')
    else:
        print(f'WARNING: Drive too full for the {size/1e9:.2f} GB backup ({free/1e9:.2f} GB free) -- skipped.')
    return sub

def restore_vcc_diverse():
    subprocess.run(['cp', VCC_BACKUP, '/content/vcc2020_subset_backup.tar.gz'], check=True)
    subprocess.run(['tar', '-xzf', '/content/vcc2020_subset_backup.tar.gz', '-C', DATA_DIR], check=True)
    return pd.read_csv(VCC_MANIFEST)

if os.path.exists(VCC_BACKUP) and os.path.exists(VCC_MANIFEST):
    print('Restoring VCC2020-diverse subset from Drive backup...')
    vcc_df = restore_vcc_diverse()
else:
    vcc_df = build_vcc_diverse()

print('vcc2020div is spoof-only (paired with ITW genuine in assemble):')
print(vcc_df.groupby('split').size())
print('systems per split:', vcc_df.groupby('split')['system_id'].nunique().to_dict())
train_sys = set(vcc_df[vcc_df.split == 'train']['system_id'])
heldout_sys = set(vcc_df[vcc_df.split == 'heldout']['system_id'])
assert not (train_sys & heldout_sys), 'system leak between train and heldout after restore!'
print('No system leak between train and T6 heldout.')


## Assemble train / val / test

In [ ]:
CAP_TRAIN, CAP_VAL, CAP_TEST = 2500, 400, 800
REALISTIC_TARGET_PER_CLASS = 120  # ~8x the raw genuine count (~15, 3 speakers) -- v3's original
                                    # proven multiplier, not the 20-60x used in v4-v7. That was
                                    # set before In-The-Wild/LibriSpeech existed to give this small,
                                    # narrow-diversity set SOME weight; now that there's far more
                                    # genuine-voice diversity elsewhere, pushing it harder than this
                                    # risks teaching the model an overly specific idea of "genuine"
                                    # from just 3-4 voices, worsening the exact problem v8 targets.

def family_of(src):
    s = str(src).lower()
    for key in ['asvspoof', 'fakeavceleb', 'wavefake', 'deepvoice', 'inthewild', 'realistic']:
        if key in s:
            return key
    return s

def balance_per_family(df, cap, seed=42):
    parts = []
    for fam, g in df.groupby('family'):
        gr, gf = g[g['label'] == 'genuine'], g[g['label'] == 'deepfake']
        n = min(len(gr), len(gf), cap)
        if n == 0:
            print(f'  dropping family {fam!r}: only one class present ({len(gr)} genuine / {len(gf)} deepfake)')
            continue
        parts += [gr.sample(n, random_state=seed), gf.sample(n, random_state=seed)]
    return pd.concat(parts) if parts else g.iloc[0:0]

# 1. v2 manifest: ASVspoof / FakeAVCeleb ... keep genuine vs deepfake only
df_v2 = pd.read_csv(f'{MANIFEST_DIR}/phase1_manifest.csv')
df_v2 = df_v2[df_v2['label'].isin(LABELS)].copy()
df_v2['family'] = df_v2['source'].map(family_of)
fav_mask = df_v2['family'] == 'fakeavceleb'
assert fav_mask.any(), 'No FakeAVCeleb rows found -- check the source names'
df_fav = df_v2[fav_mask]
df_v2 = df_v2[~fav_mask]

v2_train = balance_per_family(df_v2[df_v2['split'] == 'train'], CAP_TRAIN)
v2_val = balance_per_family(df_v2[df_v2['split'] == 'val'], CAP_VAL)
v2_test = balance_per_family(df_v2[df_v2['split'] == 'test'], CAP_TEST)
fav_test = balance_per_family(df_fav, CAP_TEST)

# 2. DEEP-VOICE -- training only
df_v4 = pd.read_csv(f'{MANIFEST_DIR_V4}/v4_manifest.csv')
dv = df_v4[df_v4['source'].isin(['deepvoice_real', 'deepvoice_fake'])].copy()
assert len(dv) > 0, 'No DEEP-VOICE rows in the v4 manifest'
dv['family'] = 'deepvoice'
dv_train = balance_per_family(dv, CAP_TRAIN)

# 3. In-The-Wild (speaker-split, restored from v7's backup)
itw_df['family'] = 'inthewild'
itw_train = balance_per_family(itw_df[itw_df['split'] == 'train'], CAP_TRAIN)
itw_val = balance_per_family(itw_df[itw_df['split'] == 'val'], CAP_VAL)
itw_test = balance_per_family(itw_df[itw_df['split'] == 'test'], CAP_TEST)

# 4. bank-fraud voice-changing: 3 speakers train, 1 held out -- same split as every earlier version,
#    kept identical on purpose so this exact test stays comparable across v5/v6/v7/v8.
def oversample_to(rows, target):
    d = pd.DataFrame(rows)
    reps = int(np.ceil(target / len(d)))
    return pd.concat([d] * reps, ignore_index=True).sample(target, random_state=42)

realistic_train = pd.concat([
    oversample_to([r for r in realistic_finetune_rows if r['label'] == 'genuine'], REALISTIC_TARGET_PER_CLASS),
    oversample_to([r for r in realistic_finetune_rows if r['label'] == 'deepfake'], REALISTIC_TARGET_PER_CLASS),
], ignore_index=True)
realistic_train['family'] = 'realistic'
realistic_test = pd.DataFrame(realistic_heldout_rows)
realistic_test['family'] = 'realistic'

# 5. LibriSpeech -- genuine-only, NOT run through balance_per_family (there is no deepfake side).
ls_train = librispeech_df[librispeech_df['split'] == 'train'].copy()
ls_train['family'] = 'librispeech'
ls_val = librispeech_df[librispeech_df['split'] == 'val'].copy()
ls_val['family'] = 'librispeech'
ls_test = librispeech_df[librispeech_df['split'] == 'test'].copy()
ls_test['family'] = 'librispeech'

# 6. VCC2020-diverse -- spoof-only (system-split). VCC2020's own bonafide clips proved
#    unreliable on this Kaggle mirror (many sampled IDs absent from all 3 download parts,
#    verified by hand -- not a system-diversity issue). Genuine-side generalization is already
#    solved by v8 (LibriSpeech 0/10->100%, VoxCeleb 10/10), so these unseen-system spoof clips
#    are paired with already-reliable In-The-Wild genuine clips instead, split-matched (train
#    genuine <- ITW train, val <- ITW val, T6 <- ITW test) so there's no leakage against ITW's
#    own train/val/test boundaries.
vcc_df['family'] = 'vcc2020div'
_PAIR_COLS = ['path', 'label', 'family']

def pair_spoof_with_genuine(spoof_df, genuine_pool, cap, seed=42):
    n = min(len(spoof_df), len(genuine_pool), cap)
    assert n > 0, 'vcc2020div: no spoof or no genuine available to pair -- check the download cell'
    spoof_part = spoof_df.sample(n, random_state=seed)
    genuine_part = genuine_pool.sample(n, random_state=seed).copy()
    genuine_part['label'] = 'genuine'
    genuine_part['family'] = 'vcc2020div'
    return pd.concat([spoof_part[_PAIR_COLS], genuine_part[_PAIR_COLS]], ignore_index=True)

itw_train_genuine_pool = itw_df[(itw_df['split'] == 'train') & (itw_df['label'] == 'genuine')]
itw_val_genuine_pool = itw_df[(itw_df['split'] == 'val') & (itw_df['label'] == 'genuine')]
itw_test_genuine_pool = itw_df[(itw_df['split'] == 'test') & (itw_df['label'] == 'genuine')]

vcc_train = pair_spoof_with_genuine(vcc_df[vcc_df['split'] == 'train'], itw_train_genuine_pool, CAP_TRAIN)
vcc_val = pair_spoof_with_genuine(vcc_df[vcc_df['split'] == 'val'], itw_val_genuine_pool, CAP_VAL)
vcc_heldout_test = pair_spoof_with_genuine(vcc_df[vcc_df['split'] == 'heldout'], itw_test_genuine_pool, CAP_TEST)

cols = ['path', 'label', 'family']
train_df = pd.concat([d[cols] for d in (v2_train, dv_train, itw_train, realistic_train, ls_train, vcc_train)], ignore_index=True)
# ls_val and vcc_val are included here on purpose -- without them, checkpoint selection (which
# reads val_df) has zero signal on whether either fix is actually working during training.
val_df = pd.concat([d[cols] for d in (v2_val, itw_val, ls_val, vcc_val)], ignore_index=True)

TEST_SETS = {
    'T1 In-The-Wild, unseen speakers': itw_test[cols],
    'T2 bank-fraud, unseen speaker (20 files)': realistic_test[cols],
    'T3 ASVspoof, in-domain': v2_test[cols],
    'T4 FakeAVCeleb, unseen dataset': fav_test[cols],
    'T5 LibriSpeech, unseen speakers (genuine-only)': ls_test[cols],
    'T6 VCC2020, unseen voice-conversion systems': vcc_heldout_test[cols],
}

print('\nTRAIN composition (clips per family and class):')
print(train_df.groupby(['family', 'label']).size().unstack(fill_value=0))
print('\nVAL:', len(val_df), ' TESTS:', {k: len(v) for k, v in TEST_SETS.items()})
for name, d in TEST_SETS.items():
    assert not (set(d['path']) & set(train_df['path'])), f'test/train path overlap in {name}'
print('No train/test file overlap.')


In [ ]:
def preflight(df, name, per_family=25):
    bad, checked = 0, 0
    for fam, g in df.groupby('family'):
        for p in g['path'].sample(min(per_family, len(g)), random_state=0):
            checked += 1
            try:
                a = load_mono_16k(p)
                if len(a) < 1600 or np.abs(a).max() < 1e-4:
                    bad += 1
            except Exception:
                bad += 1
    print(f'{name}: {bad}/{checked} unreadable or silent')
    return bad, checked

tot_bad = tot = 0
for nm, d in [('train', train_df), ('val', val_df)] + list(TEST_SETS.items()):
    b, c = preflight(d, nm)
    tot_bad += b
    tot += c
assert tot_bad / max(tot, 1) < 0.02, 'Too many unreadable clips -- fix before training'
print('Preflight OK.')


## Train v9: fine-tune from v8 (not from scratch)

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import csv

V9_BATCH_SIZE = 8
V9_GRAD_ACCUM = 4
V9_EPOCHS = 3
V9_LR = 1e-5         # same proven nudge rate -- recognizing unfamiliar voice-conversion systems is
                     # a genuinely new behavior, same reasoning as v8's LibriSpeech fix.
PATIENCE = 2

model = AudioClassifier(num_classes=2).to(device)
v7_export = f'{PROJECT_DIR}/export_v7/wav2vec2_deepfake_classifier.pt'
v8_export = f'{PROJECT_DIR}/export_v8/wav2vec2_deepfake_classifier.pt'
v8_ckpt = f'{CKPT_DIR_V8}/best.pt'
start_from = v8_export if os.path.exists(v8_export) else v8_ckpt
model.load_state_dict(torch.load(start_from, map_location=device))
print('Starting from v8 weights:', start_from)

train_loader = DataLoader(AudioDatasetV7(train_df, train=True, augment=True), batch_size=V9_BATCH_SIZE,
                          shuffle=True, collate_fn=collate, num_workers=2)
val_loader = DataLoader(AudioDatasetV7(val_df, train=False), batch_size=V9_BATCH_SIZE,
                        shuffle=False, collate_fn=collate, num_workers=2)

counts = train_df['label'].value_counts()
weights = torch.tensor([1.0 / counts.get(l, 1) for l in LABELS], dtype=torch.float32)
weights = (weights / weights.sum() * len(LABELS)).to(device)
print('Class weights:', dict(zip(LABELS, weights.tolist())))

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW(model.parameters(), lr=V9_LR)
total_steps = (len(train_loader) // V9_GRAD_ACCUM) * V9_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)

def evaluate_val(loader):
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for waveforms, labels in loader:
            logits = model(waveforms.to(device))
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            labels_all.extend(labels.numpy())
    return f1_score(labels_all, preds, average='macro')

LOCAL_CKPT_DIR = '/content/checkpoints_v9_local'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

def save_ckpt_verified(obj, filename):
    local_path = f'{LOCAL_CKPT_DIR}/{filename}'
    drive_path = f'{CKPT_DIR_V9}/{filename}'
    torch.save(obj, local_path)
    local_size = os.path.getsize(local_path)
    try:
        shutil.copy2(local_path, drive_path)
        drive_ok = os.path.exists(drive_path) and os.path.getsize(drive_path) == local_size
    except OSError as e:
        print(f'  [ckpt] Drive write error: {e}')
        drive_ok = False
    print(f'  [ckpt] {filename}: local={local_size/1e6:.1f}MB  drive_saved={drive_ok}')
    if not drive_ok:
        print(f'  [ckpt] WARNING: Drive copy failed/incomplete for {filename} -- local copy at {local_path}')
    return drive_ok

ckpt_path = f'{CKPT_DIR_V9}/latest.pt'
log_path = f'{LOG_DIR_V9}/training_log.csv'
start_epoch, best_val_f1, epochs_no_improve = 0, -1.0, 0

pre_val_f1 = evaluate_val(val_loader)
best_val_f1 = pre_val_f1
print(f'val macro-F1 BEFORE v9 fine-tuning (i.e. v8 itself, on this val set): {pre_val_f1:.4f}')

if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    start_epoch = state['epoch'] + 1
    best_val_f1 = state.get('best_val_f1', pre_val_f1)
    epochs_no_improve = state.get('epochs_no_improve', 0)
    print(f'Resuming v9 from epoch {start_epoch} (best val macro-F1 so far: {best_val_f1:.4f})')
else:
    print('No checkpoint found -- starting v9 fresh from epoch 0.')

if not os.path.exists(log_path):
    with open(log_path, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_macro_f1'])

for epoch in range(start_epoch, V9_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    for step, (waveforms, labels) in enumerate(tqdm(train_loader, desc=f'v9 epoch {epoch}')):
        waveforms, labels = waveforms.to(device), labels.to(device)
        loss = criterion(model(waveforms), labels) / V9_GRAD_ACCUM
        loss.backward()
        running_loss += loss.item() * V9_GRAD_ACCUM
        if (step + 1) % V9_GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    train_loss = running_loss / len(train_loader)
    val_f1 = evaluate_val(val_loader)
    print(f'epoch {epoch}: train_loss={train_loss:.4f}  val_macro_f1={val_f1:.4f}')
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, train_loss, val_f1])

    if val_f1 > best_val_f1:
        best_val_f1, epochs_no_improve = val_f1, 0
        save_ckpt_verified(model.state_dict(), 'best.pt')
    else:
        epochs_no_improve += 1

    save_ckpt_verified({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch,
                        'best_val_f1': best_val_f1, 'epochs_no_improve': epochs_no_improve}, 'latest.pt')

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'\nv9 training done. Best val macro-F1: {best_val_f1:.4f}  (v8 baseline was {pre_val_f1:.4f})')
print('Files on Drive:', sorted(os.listdir(CKPT_DIR_V9)))


## Evaluate on data the model never trained on (v9 vs v8 vs v7 vs v5)

In [ ]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def predict_probs(m, df, batch_size=16):
    m.eval()
    loader = DataLoader(AudioDatasetV7(df, train=False), batch_size=batch_size, shuffle=False,
                        collate_fn=collate, num_workers=2)
    out = []
    for waveforms, _ in tqdm(loader, desc='predicting', leave=False):
        out.append(F.softmax(m(waveforms.to(device)), dim=-1).cpu().numpy())
    return np.concatenate(out)

def binary_metrics(y_true, p_fake, pred_fake):
    y_true, pred_fake = np.asarray(y_true), np.asarray(pred_fake)
    real, fake = y_true == 0, y_true == 1
    return {
        'n': len(y_true),
        'real_ok': (~pred_fake[real]).mean() if real.any() else np.nan,
        'fake_ok': pred_fake[fake].mean() if fake.any() else np.nan,
        'acc': (pred_fake == fake).mean(),
        'auc': roc_auc_score(y_true, p_fake) if real.any() and fake.any() else np.nan,
    }

def passes(row):
    real_ok_pass = row['real_ok'] >= 0.8
    fake_ok_pass = row['fake_ok'] >= 0.8 or pd.isna(row['fake_ok'])   # genuine-only test sets: don't require a fake side
    return bool(real_ok_pass and fake_ok_pass)

# v9: best checkpoint (fall back to latest if best is missing, same as v7/v8's pattern)
best_path = f'{CKPT_DIR_V9}/best.pt'
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
else:
    model.load_state_dict(torch.load(f'{CKPT_DIR_V9}/latest.pt', map_location=device)['model'])
model.eval()

# v8 reference (binary, same architecture)
v8_model = None
if os.path.exists(v8_export):
    v8_model = AudioClassifier(num_classes=2).to(device)
    v8_model.load_state_dict(torch.load(v8_export, map_location=device))
    v8_model.eval()

# v7 reference (binary, same architecture)
v7_model = None
if os.path.exists(v7_export):
    v7_model = AudioClassifier(num_classes=2).to(device)
    v7_model.load_state_dict(torch.load(v7_export, map_location=device))
    v7_model.eval()

# v5 reference (3-class: 0=genuine, 1=ai_generated, 2=deepfake)
v5_model = None
v5_path = f'{CKPT_DIR_V5}/latest.pt'
if os.path.exists(v5_path):
    v5_model = AudioClassifier(num_classes=3).to(device)
    v5_model.load_state_dict(torch.load(v5_path, map_location=device)['model'])
    v5_model.eval()

rows = []
for name, d in TEST_SETS.items():
    y = d['label'].map(LABEL_TO_IDX).values
    p9 = predict_probs(model, d)
    m9 = binary_metrics(y, p9[:, 1], p9.argmax(1) == 1)
    rows.append({'test set': name, 'model': 'v9', **m9})
    if v8_model is not None:
        p8 = predict_probs(v8_model, d)
        m8 = binary_metrics(y, p8[:, 1], p8.argmax(1) == 1)
        rows.append({'test set': name, 'model': 'v8', **m8})
    if v7_model is not None:
        p7 = predict_probs(v7_model, d)
        m7 = binary_metrics(y, p7[:, 1], p7.argmax(1) == 1)
        rows.append({'test set': name, 'model': 'v7', **m7})
    if v5_model is not None:
        p5 = predict_probs(v5_model, d)
        m5 = binary_metrics(y, 1 - p5[:, 0], p5.argmax(1) != 0)
        rows.append({'test set': name, 'model': 'v5', **m5})

res = pd.DataFrame(rows)
res['PASS (>=80%)'] = res.apply(passes, axis=1)
pd.set_option('display.width', 200)
print(res.round(3).to_string(index=False))
res.to_csv(f'{PROJECT_DIR}/v9_vs_v8_v7_v5_results.csv', index=False)

v9_rows = res[res['model'] == 'v9']
v8_rows = res[res['model'] == 'v8']
print('\nv9 passes on', int(v9_rows['PASS (>=80%)'].sum()), 'of', len(v9_rows), 'test sets.')
t6_v9 = v9_rows[v9_rows['test set'].str.startswith('T6')]
t6_v8 = v8_rows[v8_rows['test set'].str.startswith('T6')]
if len(t6_v9) and len(t6_v8):
    print(f"T6 unseen VC systems (the thing this run targeted): v8 was "
          f"{t6_v8['acc'].iloc[0]:.1%} (chance); v9 gets {t6_v9['acc'].iloc[0]:.1%}.")
print('Agreed bar before this run: hold T1-T5 steady vs v8, get T6 clearly above chance (50%)'
      ' -- not 100%. Report the real number, whatever it is.')
print('Note: T2 has only 5 real clips -- one clip moves it by 20 points; treat it as a sanity check.')


## Export v9 (only worth keeping if the results above meet the agreed bar)

In [ ]:
import json as json_lib
export_dir_v9 = f'{PROJECT_DIR}/export_v9'
os.makedirs(export_dir_v9, exist_ok=True)
local_export = '/content/export_v9'
os.makedirs(local_export, exist_ok=True)
torch.save(model.state_dict(), f'{local_export}/wav2vec2_deepfake_classifier.pt')
with open(f'{local_export}/label_map.json', 'w') as f:
    json_lib.dump({'labels': LABELS, 'model_name': 'facebook/wav2vec2-base',
                   'note': 'v9 = v8 fine-tuned with a system-diverse VCC2020 voice-conversion source '
                           '(~40 systems, system-level split), aimed at closing the unseen-VC-system '
                           'generalization gap found by the T6 stress test (v8: 30/59, chance level)'}, f, indent=2)
size = os.path.getsize(f'{local_export}/wav2vec2_deepfake_classifier.pt')
free = shutil.disk_usage(PROJECT_DIR).free
if size * 1.05 < free:
    shutil.copy2(f'{local_export}/wav2vec2_deepfake_classifier.pt', export_dir_v9)
    shutil.copy2(f'{local_export}/label_map.json', export_dir_v9)
    print('Exported to Drive:', export_dir_v9)
else:
    print(f'Drive too full for the export ({free/1e9:.2f} GB free) -- NOT saved to Drive.')

shutil.make_archive('/content/export_v9', 'zip', local_export)
from google.colab import files
files.download('/content/export_v9.zip')
